<a href="https://colab.research.google.com/github/jarekwan/praca_inzynierska/blob/main/split_time.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/ml_project', exist_ok=True)
print("folder ready")

import sys
sys.path.append('/content/drive/MyDrive/ml_project')

Mounted at /content/drive
folder ready


In [2]:
%%writefile /content/drive/MyDrive/ml_project/split_time.py
# -*- coding: utf-8 -*-
import os
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

target_dir = "/content/drive/MyDrive/ml_project"

# wejścia
input_x_scaled = os.path.join(target_dir, "X_scaled.pkl")   # jeśli istnieje, używamy tego
input_x = os.path.join(target_dir, "X.pkl")                 # fallback
input_y = os.path.join(target_dir, "y.pkl")

# konfiguracja z ui
choice_file = os.path.join(target_dir, "test_size.txt")

# wyjścia
output_x_train = os.path.join(target_dir, "X_train.pkl")
output_x_test = os.path.join(target_dir, "X_test.pkl")
output_y_train = os.path.join(target_dir, "y_train.pkl")
output_y_test = os.path.join(target_dir, "y_test.pkl")

# ---------------------------------------------------------
# split_time_ui
# ---------------------------------------------------------
def split_time_ui():

    # ui do wyboru procentu danych testowych
    label = widgets.Label("select test size (e.g. 0.2 = 20% for test set):")
    textbox = widgets.Text(value="0.2")

    btn = widgets.Button(description="confirm")
    out = widgets.Output()

    def on_click(b):
        out.clear_output()

        val = textbox.value.strip()

        # zapisujemy wartość jako tekst
        with open(choice_file, "w") as f:
            f.write(val + "\n")

        with out:
            print("saved:", choice_file)
            print("test size:", val)

    btn.on_click(on_click)

    display(widgets.VBox([
        label,
        textbox,
        btn,
        out
    ]))

# ---------------------------------------------------------
# split_time
# ---------------------------------------------------------
def split_time():

    # decydujemy, które x wczytać
    if os.path.exists(input_x_scaled):
        x = pd.read_pickle(input_x_scaled)
    elif os.path.exists(input_x):
        x = pd.read_pickle(input_x)
    else:
        raise FileNotFoundError("no X.pkl or X_scaled.pkl found")

    if not os.path.exists(input_y):
        raise FileNotFoundError("y.pkl not found")

    y = pd.read_pickle(input_y)

    if not os.path.exists(choice_file):
        raise FileNotFoundError("test_size.txt not found")

    with open(choice_file, "r") as f:
        test_size = float(f.read().strip())

    # obliczamy indeks podziału
    n = len(x)
    split_point = int(n * (1 - test_size))

    # chronologiczny podział
    x_train = x.iloc[:split_point].copy()
    x_test = x.iloc[split_point:].copy()
    y_train = y.iloc[:split_point].copy()
    y_test = y.iloc[split_point:].copy()

    # zapis wyników
    x_train.to_pickle(output_x_train)
    x_test.to_pickle(output_x_test)
    y_train.to_pickle(output_y_train)
    y_test.to_pickle(output_y_test)

    print("saved:", output_x_train)
    print("saved:", output_x_test)
    print("saved:", output_y_train)
    print("saved:", output_y_test)
    print("split completed")

    return x_train, x_test, y_train, y_test

Writing /content/drive/MyDrive/ml_project/split_time.py
